In [5]:
#declaring the necessary librarys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.optimize import curve_fit

Part 1 of 8: Analytical solution for two boxes collisions ($M \geq m$)

We're told that the collisions are perfectly elastic. Therefore, total kinetic energy and total momentum are conserved at the collision. The conservation equations are:

$$ \frac{M V_b^2}{2} + \frac{m v_b^2}{2} = \frac{M V_a^2}{2} + \frac{m v_a^2}{2} $$

$$ M V_b + m v_b = M V_a + m v_a $$

We have to find and return the final velocities ($V_a, v_a$) and express the masses in terms of the given mass ratio $R = M/m$. Resulting in:

$$ V_a = \frac{R-1}{R+1}V_b + \frac{2}{R+1}v_b $$

$$ v_a = \frac{2R}{R+1}V_b + \frac{1-R}{R+1}v_b $$

Implementing the analytical asked solution using a function that takes the initial velocities and $R$ as arguments and returns the post-collision velocities.

In [6]:
def box_collision(Vb, vb, R):
 
    #using the velocities formula in expression of the given mass ratio
    Va = ((R - 1) / (R + 1)) * Vb + (2 / (R + 1)) * vb
    va = (2 * R / (R + 1)) * Vb + ((1 - R) / (R + 1)) * vb
    
    return Va, va

Part 2 of 8: Simulating the sequence of collisions

Now we consider the collision between the smaller box and the wall. The wall is assumed to have a much larger mass than the box, so its velocity does not change. Since the collision is perfectly elastic, the box keeps its speed and only its direction is reversed:

$$ v_a = -v_b $$

Using this and the function from Part 1, we write NumberOfCollisions(Vi, R), which simulates the whole sequence of collisions. The function works as follows:

- The big box starts with velocity $V_i$ and the small box is at rest ($v = 0$). The counter Nc starts at 0 and the list AllVelocities stores the pair $[V, v]$ after every collision.
- Since we don't know the number of collisions beforehand, we use a `while` loop. Another box-box collision is only possible if the big box (on the left) moves faster than the small box, so the loop condition is $V > v$.
- Inside the loop we first apply the box-box collision with box_collision, add 1 to Nc and store the new velocities.
- After that, if the small box moves to the right ($v > 0$), it will hit the wall. We reverse its velocity ($v = -v$), add 1 to Nc (box-wall collisions also count) and store the new velocities.
- When $V \leq v$ the boxes move apart and can no longer collide, so the loop ends.

The function returns the total number of collisions Nc and the list AllVelocities.

In [7]:
#defining the function 
def NumberOfCollisions(Vi, R):
    V, v = Vi, 0.0          
    Nc = 0                  
    AllVelocities = []      

    #looping due to the unknown number of colissions 
    while V > v:            
        #calling the previous function
        V, v = box_collision(V, v, R)
        Nc += 1
        AllVelocities.append([V, v])

        # if the small box moves right it hits the wall and reverses (collision effect)
        if v > 0:
            v = -v
            Nc += 1
            AllVelocities.append([V, v])

    return Nc, AllVelocities

To check that the function works, we test it for several mass ratios. For $R = 1$ we expect exactly 3 collisions (the boxes exchange velocities, the small box bounces off the wall and they collide once more), and for $R = 100$ we expect 31.

In [8]:
#loop for R with each inttended value
for R in [1, 2, 10, 100]:
    Nc, AllVelocities = NumberOfCollisions(1, R)
    print(f"R = {R:4}  ->  Nc = {Nc}")

R =    1  ->  Nc = 3
R =    2  ->  Nc = 5
R =   10  ->  Nc = 10
R =  100  ->  Nc = 31


Part 3 of 8: Does the number of collisions depend on the initial velocity?

For a fixed mass ratio $R = M/m$, we want to know if the total number of collisions $N_c$ depends on the initial velocity $V_i$. To find out, we keep $R = 100$ fixed and call NumberOfCollisions(Vi, R) for several very different values of $V_i$, and compare the results.

In [9]:
R = 100   # fixed mass ratio
Vi_values = [0.01, 0.1, 1, 10, 100, 1000]

#loop over the initial velocities, keeping R fixed
for Vi in Vi_values:
    Nc, AllVelocities = NumberOfCollisions(Vi, R)
    print(f"Vi = {Vi:8} m/s  ->  Nc = {Nc}")

Vi =     0.01 m/s  ->  Nc = 31
Vi =      0.1 m/s  ->  Nc = 31
Vi =        1 m/s  ->  Nc = 31
Vi =       10 m/s  ->  Nc = 31
Vi =      100 m/s  ->  Nc = 31
Vi =     1000 m/s  ->  Nc = 31


Observation: $N_c$ is exactly the same for all the values of $V_i$ we tried (for $R = 100$ it is always 31). The initial velocity does not affect the number of collisions.

Dimensional analysis: $N_c$ is a pure number and $R = M/m$ is dimensionless, while $V_i$ has dimensions of velocity (m/s). The only other quantities in the problem are the masses (kg), and no length or time scale appears (the wall position does not matter). Without another quantity with dimensions of length or time, $V_i$ cannot be combined with anything to form a dimensionless number. Since $N_c$ must be dimensionless, it can only be a function of $R$: $N_c = N_c(R)$. This also makes physical sense: changing $V_i$ just rescales all the velocities by the same factor, and the collision formulas are linear in the velocities, so the sequence of collisions is the same.

Part 4 of 8: Geometrical view of the collisions

During an elastic collision the total kinetic energy is conserved, so all the pairs of velocities $(V, v)$ must satisfy

$$ \frac{V^2}{V_i^2} + \frac{v^2}{R V_i^2} = 1 $$

which is an ellipse in the velocity space centered at $(0,0)$ with semi-axes $a = V_i$ (along $V$) and $b = V_i\sqrt{R}$ (along $v$).

We plot this ellipse with `Ellipse` from `matplotlib.patches`. Since `width` and `height` are the full lengths of the axes, we pass $2a$ and $2b$. On top of it we plot the list `AllVelocities` returned by `NumberOfCollisions`, adding the initial point $(V_i, 0)$ at the beginning so the trajectory starts where the system starts. We do this for $R = 1, 2, 10$ and $100$.